## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Fall 2025


# Homework 1: Let's GOOOOO!

- **100 points [7.5% of your final grade]**
- **Due Tuesday, September 14 by 11:59pm**

***Goals of this homework:***
1. Collect data from the web, clean it, and then make some observations based on exploratory data analysis
2. Understand and implement the classic apriori algorithm and extensions to find the association rules in a movie rating dataset
3. Push yourself by using Spark to find association rules

***Submission instructions:***

You should post your notebook to Canvas (look for the homework 1 assignment there). Please name your submission **your-uin_hw1.ipynb**, so for example, my submission would be something like **555001234_hw1.ipynb**. Your notebook should be fully executed when you submit ... so run all the cells for us so we can see the output, then submit that.

***Late Days:***

As a reminder, you begin the semester with five late days. You may use as many as you like. There is no need to alert us to how many late days you are using. Just submit and we will make note of it. Also remember that once your late days are used up, homeworks will receive a 0.

***Collaboration and AI Assistance declaration:***

If you worked with someone on this homework, please be sure to mention that. Remember to include citations to any sources you use in the homework. Also tell us what AI assistant you used and how you used it.

## (REQUIRED) Collaboration and AI Assistance Declaration

### Collaboration Declaration:

*your response goes here*

### AI Assistance Declaration:
gpt4.1 vscode chat for helper functions or quick syntax on longer operations

In [37]:
# for out, i in [('changed', 'changing'), ('delta', 'triangle'),]:
#     ufo_raw['shape'].replace(i, out, inplace=True)
# print(ufo.loc[ufo['country'].isna() | (ufo['country'].str.strip() == '') | (ufo['country'].str.strip() == 'nan'), 'city'].value_counts())
# len(ufo_raw)
ufo_fe['duration_bucket'].value_counts()
# print(f'{min(ufo_fe['year'])}, {max(ufo_fe['year'])}')

duration_bucket
5m-30m    25998
1m-5m     21535
<10s      17166
10s-1m    13799
1h-6h      4973
30m-1h     4667
6h-24h      308
>24h        230
Name: count, dtype: int64


## (45 points) Part 1: UFO Sightings — Data Ingestion, Cleaning, and Feature Engineering

**Dataset:** `ufos.csv`

Detected columns: `datetime`, `city`, `state`, `country`, `shape`, `duration (seconds)`, `duration (hours/min)`, `comments`, `date posted`, `latitude`, `longitude`, and possibly extra unnamed columns.

**Goal:** Load the data, diagnose issues, clean/standardize it, and derive basic features to support downstream mining. You will probably want to use `pandas` for this.



### (5pts) Part 1a: Load and sanity-check the raw data

Tasks:
1. Load `ufos.csv` into a DataFrame named `ufo_raw`.
2. Display 5 random rows and `ufo_raw.info()`.
3. Briefly report the number of rows/columns and any obviously empty columns.


In [12]:
import pandas as pd
pd.set_option('display.max_rows', None)

# Read columns as string first
ufo_raw = pd.read_csv(
    'ufos.csv',
    dtype=str,
    parse_dates=['datetime', 'date posted'],
    on_bad_lines='skip'
)

# Display 5 random rows and info
from random import randint
display(ufo_raw.sample(5, random_state=randint(0, 10000)))
ufo_raw.info()

def display_info(df):
    df.info()
    display(df.describe(include='all'))

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
70275,7/7/2004 21:30,dallas,tx,us,fireball,15,15 seconds,fireball over Dallas,1089244800000000000,32.7833333,-96.8000000
18540,1/20/2007 20:00,wheatley (canada),on,ca,triangle,10,10 seconds,4 dim lights&#44 travelling in a straight line...,1170288000000000000,42.1,-82.45
36929,3/24/2009 21:40,sand springs,ok,us,other,300,5 minutes,Line of 5 lights blinking in a pattern over Sh...,1239667200000000000,36.1397222,-96.1086111
33118,2/8/2003 21:00,burbank,ca,us,fireball,60,1 minute,5th sighting in burbank,1047340800000000000,34.1808333,-118.3080556
13336,1/12/2008 11:00,st. petersburg,fl,us,changing,10800,3 hours,Strange changing object with different color l...,1200873600000000000,27.7705556,-82.6794444


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88679 entries, 0 to 88678
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   datetime              88679 non-null  object
 1   city                  88679 non-null  object
 2   state                 81270 non-null  object
 3   country               76314 non-null  object
 4   shape                 85757 non-null  object
 5   duration (seconds)    88677 non-null  object
 6   duration (hours/min)  85660 non-null  object
 7   comments              88644 non-null  object
 8   date posted           88679 non-null  object
 9   latitude              88679 non-null  object
 10  longitude             88679 non-null  object
dtypes: object(11)
memory usage: 7.4+ MB


`datetime:`, `city:str`, `state`, `country`, `shape`, `duration (seconds)`, `duration (hours/min)`, `comments`, `date posted`, `latitude`, `longitude`


### (5pts) Part 1b: Clean up datatypes & columns

Create a cleaned DataFrame `ufo`:

- Drop fully-empty or irrelevant columns (e.g., unnamed columns).
- Parse `datetime` to `datetime64[ns]` (`errors='coerce'`).
- Coerce `duration (seconds)`, `latitude`, `longitude` to numeric.
- Lowercase/trim `city`, `state`, `country`, `shape`.
- Remove rows with impossible coordinates (lat ∉ [-90,90], lon ∉ [-180,180]).
- Drop exact duplicates based on a reasonable subset (document your choice).

Provide a short markdown note explaining your choices.


In [13]:
import re
from datetime import datetime, timedelta

# datetime values with 24:00 get dropped when using pd.to_datetime
def fix_24_hour(dt_str):
    # Match date and time with 24:00
    match = re.match(r'(\d{1,2}/\d{1,2}/\d{2,4}) 24:00', str(dt_str))
    if match:
        # Parse date, add one day, set time to 00:00
        date_part = match.group(1)
        try:
            date_obj = datetime.strptime(date_part, '%m/%d/%Y')
        except ValueError:
            date_obj = datetime.strptime(date_part, '%m/%d/%y')
        new_dt = date_obj + timedelta(days=1)
        return new_dt.strftime('%m/%d/%Y 00:00')
    return dt_str

ufo = pd.DataFrame(ufo_raw)
# make 24:00 datetimes readable to pandas
ufo['datetime'] = ufo['datetime'].apply(fix_24_hour)
ufo['datetime'] = pd.to_datetime(ufo['datetime'], errors='coerce')

# Convert numeric columns, coercing errors to NaN
for col in ['duration (seconds)', 'latitude', 'longitude']:
    ufo[col] = pd.to_numeric(ufo[col], errors='coerce')
ufo = ufo[
    (ufo['latitude'] >= -90) & (ufo['latitude'] <= 90) &
    (ufo['longitude'] >= -180) & (ufo['longitude'] <= 180)
]
# Replace latitude and longitude values of 0 with NaN
ufo.loc[ufo['latitude'] == 0, 'latitude'] = None
ufo.loc[ufo['longitude'] == 0, 'longitude'] = None


# Convert other columns to desired types if needed
for col in ['city', 'state', 'country', 'shape', 'duration (hours/min)', 'comments']:
    ufo[col] = ufo[col].astype(str)
# Replace 'yk' with 'yt' in the 'state' column for the more commonly abv yukon territory
ufo['state'] = ufo['state'].replace('yk', 'yt')
# Set 'state' to 'vi' and 'pr' and 'us' for virgin islands and puerto rico
for x, y in [('virgin islands', 'vi'), ('puerto rico', 'pr')]:
    ufo.loc[
        (ufo['city'].str.lower().str.contains(x)) &
        ((ufo['state'].str.strip() == '') | (ufo['state'].str.lower() == 'nan')),
        'state'
    ] = y
    ufo.loc[
        (ufo['city'].str.lower().str.contains(x)) &
        ((ufo['country'].str.strip() == '') | (ufo['country'].str.lower() == 'nan')),
        'country'
    ] = 'us'
# for the more common country codes left blank
for x, y in [('canada', 'ca'), ('new zealand', 'nz'), ('south africa', 'za'), ('singapore', 'sg'), ('ireland', 'ie'), ('(italy)', 'it'), ('(mexico)', 'mx'), ('(uk/england)', 'gb'), ('(russia)', 'ru'), ('(lithuania)', 'lt'), ('(romania)', 'ro'), ('(india)', 'in'), ('iran', 'ir'), ('(iraq)', 'iq'), ('malaysia', 'myq'), ('australia', 'au'),('china', 'cn'), ('spain', 'es'), ('poland', 'pl'), ('greece', 'gr'), ('brazil', 'br'), ('portugal', 'pt'), ('south korea', 'kr'), ('japan', 'jp'), ('philippines', 'ph')]:
    ufo.loc[ # regex false because I want to include some parenthesis in city field
        (ufo['city'].str.lower().str.contains(x, regex=False)) &
        ((ufo['country'].str.strip() == '') | (ufo['country'].str.lower() == 'nan')),
        'country'
    ] = y
# List of two-letter US state abbreviations
us_states = [
    'al', 'ak', 'az', 'ar', 'ca', 'co', 'ct', 'de', 'fl', 'ga', 'hi', 'id', 'il', 'in', 'ia', 'ks', 'ky', 'la', 'me',
    'md', 'ma', 'mi', 'mn', 'ms', 'mo', 'mt', 'ne', 'nv', 'nh', 'nj', 'nm', 'ny', 'nc', 'nd', 'oh', 'ok', 'or', 'pa',
    'ri', 'sc', 'sd', 'tn', 'tx', 'ut', 'vt', 'va', 'wa', 'wv', 'wi', 'wy', 'dc'
]
# Set country to 'us' if state is a US abbreviation and country is missing or NaN
ufo.loc[
    (ufo['state'].str.lower().isin(us_states)) & (ufo['country'] == 'nan'),
    'country'
] = 'us'

ufo.info()
# duplicates = ufo[ufo.duplicated(keep=False)]
# print(duplicates)
# > no duplicates found

<class 'pandas.core.frame.DataFrame'>
Index: 88678 entries, 0 to 88678
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              88678 non-null  datetime64[ns]
 1   city                  88678 non-null  object        
 2   state                 88678 non-null  object        
 3   country               88678 non-null  object        
 4   shape                 88678 non-null  object        
 5   duration (seconds)    88676 non-null  float64       
 6   duration (hours/min)  88678 non-null  object        
 7   comments              88678 non-null  object        
 8   date posted           88678 non-null  object        
 9   latitude              87184 non-null  float64       
 10  longitude             87184 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 8.1+ MB


### (5pts) Part 1c: Be thankful!

Now take a look at the `duration (hours/min)` column. In a previous version of this homework, students spent upwards of 10-20 hours just cleaning the values in this column. Luckily for you, the rest of this assignment relies on the much nicer `duration (seconds)` column. For this question, we'd like you to just extract how ever many versions of durations reported in *minutes* you can from the `duration (hours/min)` column. In other words, find as many different variations of anything that could be reasonably interpreted as a minute-like duration. Examples include:

* several minutes
* x to y minutes
* x minutes
* x min.
* x mins.
* and so on ...


In [14]:
print(ufo['duration (hours/min)'].value_counts())
print(f'{len(ufo['duration (hours/min)'].value_counts())} different variations (including nan)')

duration (hours/min)
5 minutes                             4796
2 minutes                             3538
10 minutes                            3383
1 minute                              3118
nan                                   3019
3 minutes                             2553
30 seconds                            2350
15 minutes                            2105
10 seconds                            2016
5 seconds                             1836
20 minutes                            1467
1 hour                                1353
30 minutes                            1353
15 seconds                            1194
20 seconds                            1149
3 seconds                              945
4 minutes                              918
5 min                                  858
2 seconds                              699
10 min                                 662
2 hours                                661
2-3 minutes                            619
2 min                            

In [15]:
# Replace any sequence of digits with 'x' in the 'duration (hours/min)' column and create a new Series
duration_x = ufo['duration (hours/min)'].str.replace(r'\d+', 'x', regex=True)
print(duration_x.value_counts())
print(f'{len(duration_x.value_counts())} different variations when we remove specific numbers')
# If I were to try to clean this column I would pull the first number, scan for 'm','h', then 's' and set the value to num,num*60,num/60 respectively
# this would lose accuracy for fractioned or interval entries, but would give us something close to work with

duration (hours/min)
x minutes                          22776
x seconds                          13190
x min                               4779
x-x minutes                         3695
x minute                            3199
nan                                 3019
x-x seconds                         2404
x min.                              1835
x mins                              1787
x sec                               1601
x hour                              1360
x hours                             1227
x sec.                               987
xmin                                 972
x:x                                  885
x                                    683
x-x min                              592
unknown                              528
seconds                              525
x second                             410
x secs                               385
x to x minutes                       362
xminutes                             350
x-x min.                            

In [16]:
# Only pull values from 'duration (hours/min)' that contain 'm', then replace digits with 'x'
duration_x_m = ufo.loc[ufo['duration (hours/min)'].str.contains('m', case=False, na=False), 'duration (hours/min)'].str.replace(r'\d+', 'x', regex=True)
print(duration_x_m.value_counts())
print(f'{len(duration_x_m.value_counts())} different variations when we only include values believed to be in minutes')
# this approach is simple, assuming any value with 'm' is in minutes
# this will neglect values without units such as x:x (containing multiple units)

duration (hours/min)
x minutes                          22776
x min                               4779
x-x minutes                         3695
x minute                            3199
x min.                              1835
x mins                              1787
xmin                                 972
x-x min                              592
x to x minutes                       362
xminutes                             350
x-x min.                             317
x-x mins                             312
xmins                                303
~x minutes                           263
about x minutes                      224
x mins.                              206
xmin.                                183
x.x minutes                          170
one minute                           169
x - x minutes                        159
x+ minutes                           146
several minutes                      124
x-xmin                               116
few minutes                         


### (10pts) Part 1d: Feature engineering and small exploratory data analysis

Create additional columns in a new feature engineering version of the dataset called `ufo_fe`:

- `year`, `month`, `hour` from `datetime`
- `duration_log10` = log10(duration_seconds + 1)
- `duration_bucket` = categorical bins for `duration (seconds)` (you may adjust edges)
- `us_only` = 1 if `country == 'us'` else 0

Then produce:
- Value counts of `shape` (top 10).
- A table of sightings by `year` (counts).
- A `state × shape` table (top 10 states by sample size).


In [ ]:
from math import log10
ufo_fe = pd.DataFrame(ufo)
ufo_fe['year'] = ufo_fe['datetime'].dt.year
ufo_fe['month'] = ufo_fe['datetime'].dt.month
ufo_fe['hour'] = ufo_fe['datetime'].dt.hour
ufo_fe['duration_log10'] = ufo_fe['duration (seconds)'].apply(lambda x: log10(x + 1))
# Create duration_bucket column with categorical bins for 'duration (seconds)'
bins = [0, 10, 60, 300, 1800, 3600, 21600, 86400, float('inf')]
labels = ['<10s', '10s-1m', '1m-5m', '5m-30m', '30m-1h', '1h-6h', '6h-24h', '>24h']
ufo_fe['duration_bucket'] = pd.cut(ufo_fe['duration (seconds)'], bins=bins, labels=labels, right=False)
ufo_fe['us_only'] = ufo_fe['country'].apply(lambda x: 1 if x == 'us' else 0)

print(f'Top 10 shapes:')
print(ufo_fe['shape'].value_counts()[:10])
year_counts = ufo_fe['year'].value_counts().sort_index().to_frame(name='count')
print(f'\nCount of Sightings by year:')
display(year_counts)
top_states = ufo_fe[~ufo_fe['state'].str.lower().isin(['nan', '', 'none'])]['state'].value_counts().head(10).index
top_states_df = ufo_fe[ufo_fe['state'].isin(top_states)][['state', 'shape']]
top_states_df = top_states_df.reset_index(drop=True)
print(f'description about the (large) state x shape table:')
top_states_df.describe()
print(top_states,'\n',top_states_df.columns)
print(top_states_df.value_counts())


Top 10 shapes:
shape
light       17872
triangle     8489
circle       8453
fireball     6562
unknown      6319
other        6247
disk         6005
sphere       5755
oval         4119
nan          2922
Name: count, dtype: int64

Count of Sightings by year:


,count
year,
1906,1
1910,3
1914,1
1916,1
1917,1
1920,2
1925,1
1929,1
1930,2


description about the (large) state x shape table:
Index(['ca', 'wa', 'fl', 'tx', 'ny', 'az', 'il', 'pa', 'oh', 'mi'], dtype='object', name='state') 
 Index(['state', 'shape'], dtype='object')
state  shape    
ca     light        2105
wa     light         979
ca     circle        977
       triangle      939
fl     light         877
tx     light         807
ca     fireball      785
       disk          763
       other         731
       sphere        707
az     light         663
ca     unknown       663
ny     light         660
il     light         583
pa     light         545
oh     light         498
fl     fireball      481
ca     oval          454
mi     light         452
fl     circle        432
tx     triangle      410
fl     triangle      409
wa     nan           406
       fireball      397
       triangle      378
       circle        375
ny     circle        363
ca     nan           362
wa     unknown       356
il     triangle      348
tx     circle        345
ny     triangle


### (5pts) Part 1e: Observations and conclusions

In 3–6 sentences, summarize data quality and distributional patterns; reference at least one artifact from 1d.


users frequently left columns blank\
few sightings before 1995, and very few before 1950\
most sightings in the US\
much of the less common free response data (like shape) could be classified as more common labels \(round, circle, sphere|triangle,delta|light,fireball|unknown,other)


### (5pts) Part 1e: Next steps

Propose 2–3 concrete next steps (e.g., better deduplication with fuzzy text, geospatial clustering, normalization of duration text, timezone handling) to improve the quality of the data before moving on to some downstream tasks (you don't need to implement these).


I think our chosen buckets for duration(s) represent the data well, with the bulk of our data between 1m-30m (in two buckets) then 0s-1m (in two buckets).\
Combining labels for shapes and checking accuracy of longitude and latitude against location data would improve the quality of our data.\
uniform representation of time (done) was important as well as we list times as both 24:00 and 00:00 which pandas could not handle.


### (10 points) Part 1 f: Scalable & Structured Patterns

Now let's explore **network**, **geospatial**, and **time** patterns to dig a little deeper into our data.
Complete **any two** sub‑tasks below with **pandas**. For each you should provide your code, the ouput, plus some written analysis of what you find.



#### Sub-task 1. Co‑witness graph (space‑time co‑occurrence)
Bucket by geocell (`lat1=round(latitude,1)`, `lon1=round(longitude,1)`) and `datetime` floored to 15 minutes.  
Two reports in the same bucket are co‑witnessed. Build edges between reports in each bucket and summarize the graph (top buckets, approximate largest component).



#### Sub-task 2.  Geospatial hotspots
Aggregate by (`lat1`,`lon1`) and list **top‑20** cells. Optionally justify a normalization (per capita proxies or surface area) if you apply one.



#### Sub-task 3.  Shape × color co‑occurrence
Extract color keywords from `comments` (red/orange/yellow/green/blue/purple/violet/white/black/silver/gold/pink; handle “-ish” variants). Build a co‑occurrence table with `shape` or your normalized `shape_norm`. Interpret one pairing.



#### Sub-task 4. Spatio‑temporal spikes (anomaly cues)
For each geocell, create a daily count series; compute z‑scores within cell; list top‑10 spikes across all cells. Discuss plausible causes.


## (40 points) Part 2: Association Rules in Movie Rating Behaviors

For the second part of this homework, we're going to examine movies using our understanding of association rules, to find movies that "go together". For this part, you will implement the apriori algorithm, and apply it to a movie rating dataset. We'll use the [MovieLens](https://grouplens.org/datasets/movielens/) dataset.

First, run the next cell to load the dataset we are going to use.

In [ ]:
import urllib3
import zipfile

http = urllib3.PoolManager()
req = http.request("GET", "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", preload_content=False)

with open("movie.zip", 'wb') as out:
  while True:
    data = req.read(4096)
    if not data:
      break
    out.write(data)
req.release_conn()

zFile = zipfile.ZipFile("movie.zip", "r")
for fileM in zFile.namelist():
  zFile.extract(fileM)

In [ ]:
!ls ml-latest-small/

In this dataset, there are four columns: `userId` is the integer ids of users, `movieId` is the integer ids of movies, `rating` is the rate of the user gives to the movie, and `timestamp` which we do not use here. Each row denotes that the user of given `userId` rated the movie of the given `movieId`. We are going to treat each user as a "basket", so you will need to collect all the movies that have been rated by a single user as a basket.

Now, you need to implement the apriori algorithm and apply it to this dataset to find association rules of user rating behaviors where:

1. Define `rating` >= 3 is "like" (that is, only consider movie ratings of 3 or higher in your baskets; you may ignore all others)
2. `minsup` == 40 (out of 600 users/baskets); we may adjust this based on the discussion on Canvas
3. `minconf` == to be determined by a discussion on Canvas. You may try several different choices, but we will converge on a good choice for everyone for the final submission.

We know there are many existing implementations of apriori online (check github for some good starting points). You are welcome to read existing codebases and let that inform your approach. Do not copy-paste any existing code. We want your code to have sufficient comments to explain your steps, to show us that you really know what you are doing. Furthermore, you should add print statements to print out the intermediate steps of your method -- e.g., the size of the candidate set at each step of the method, the size of the filtered set, and any other important information you think will highlight the method.

To help get you started, we can load the ratings with the following code snippet:

In [ ]:
import pandas as pd
# read user ratings
allRatings = pd.read_csv("ml-latest-small/ratings.csv")
allRatings

FileNotFoundError: [Errno 2] No such file or directory: 'ml-latest-small/ratings.csv'

### (15pts) Step 1: Implement Apriori Algorithm
In this section, you need to implement the Apriori algorithm, we will check the correctness of your code and we encourage efficient implementation.

In [ ]:
# your code here

### (5pts) Step 2: Print Your Association Rules

Next you should print your final association rules in the following format:

**movie_name_1, movie_name_2, ... -->
movie_name_k**

where the movie names can be fetched by joining the movieId with the file `movies.csv`. For example, one rule that you might find is:

**Matrix, The (1999),  Star Wars: Episode V - The Empire Strikes Back (1980),  Star Wars: Episode IV - A New Hope (1977),  ->
Star Wars: Episode VI - Return of the Jedi (1983)**

In [ ]:
# your code here

### (10pts) Step 3: Implement Random Sampling

We discussed in class a method to randomly sample baskets to avoid the overhead of reading the entire set of baskets (which in practice, could amount to billions of baskets). For this part, you should implement such a random sampling approach that takes a special parameter **alpha** that controls the size of the sample: e.g., alpha = 0.10 means to sample 10% of the baskets (our users, in this case).

Vary **alpha** and report the number of frequent itemsets you find and how this compares to the number of frequent itemsets in the entire dataset. What do you discover?


In [ ]:
# your code here

*your discussion here*

### (10pts) Step 4: Check for False Positives

Next you should verify that the candidate pairs you discover by random sampling are truly frequent by comparing to the itemsets you discover over the entire dataset.

For this part, consider another parameter **minsup_sample** that relaxes the minimum support threshold. For example if we want minsup = 1/100 for whole dataset, then try minsup_sample = 1/125 for the sample. This will help catch truly frequent itemsets.

Vary **minsup_sample** and report the number of frequent itemsets you find and the number of false positives you find. What do you discover?


In [ ]:
# your code here

*your discussion here*

## (5 points) Part 3: Spark-based Association Rules

So far, we have been working with a fairly small dataset. For this last question, you should use the much larger **Movies 10M** dataset: https://files.grouplens.org/datasets/movielens/ml-10m.zip

First, we need to load this larger dataset:

In [ ]:
import urllib3
import zipfile

http = urllib3.PoolManager()
req = http.request("GET", "https://files.grouplens.org/datasets/movielens/ml-10m.zip", preload_content=False)

with open("movie.zip", 'wb') as out:
  while True:
    data = req.read(4096)
    if not data:
      break
    out.write(data)
req.release_conn()

zFile = zipfile.ZipFile("movie.zip", "r")
for fileM in zFile.namelist():
  zFile.extract(fileM)

Now, see if you can write a Spark-based implementaiton of Apriori. You'll see that the Spark library MLib does contain a "Frequent Pattern Mining" implementation. You may not use this! Good luck!

In [ ]:
# your code here